# 최신 로컬 OCR 5종 — 한국어 필기 비교

API 키와 페이지당 과금 없이 공개 가중치를 Colab GPU에 내려받아 비교합니다. 기존 `handwriting_ocr_benchmark`의 동일한 S/O/A/P 필드 이미지를 사용하고, OCR 뒤 LLM 교정은 적용하지 않습니다.

| 모델 | 규모 | 라이선스 | 비고 |
|---|---:|---|---|
| PaddleOCR-VL 1.6 | 0.9B | Apache-2.0 | 다국어 문서 OCR/VLM |
| GLM-OCR | 0.9B | MIT | 한국어 포함 8개 언어, 최신 경량 후보 |
| LightOnOCR-2 | 1B | Apache-2.0 | 빠른 OCR 특화 모델, 한국어는 실험 검증 대상 |
| DeepSeek-OCR-2 | 3B | Apache-2.0 | 최신 Transformers 호환 포트 사용 |
| Qwen3-VL-4B-Instruct | 4B | Apache-2.0 | 32개 언어 OCR 일반 VLM 기준선 |

> 권장 GPU는 L4 24GB 또는 A100입니다. T4 16GB도 순차 실행은 가능하지만 3B/4B 모델이 느릴 수 있습니다.

## 1. Drive 연결과 프로젝트 찾기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_CANDIDATES = [
    Path('/content/drive/MyDrive/handwriting_ocr_benchmark_colab/handwriting_ocr_benchmark'),
    Path('/content/drive/MyDrive/handwriting_ocr_benchmark'),
    Path('/content/handwriting_ocr_benchmark_colab/handwriting_ocr_benchmark'),
    Path('/content/handwriting_ocr_benchmark'),
]
PROJECT_DIR = next((path for path in PROJECT_CANDIDATES if path.exists()), None)
assert PROJECT_DIR is not None, (
    'handwriting_ocr_benchmark를 찾지 못했습니다. PROJECT_CANDIDATES에 실제 경로를 추가하세요.'
)
for required in ('local_modern_ocr_adapters.py', 'LOCAL_OCR_MODEL_NOTES.md'):
    assert (PROJECT_DIR / required).exists(), f'{required}가 프로젝트 루트에 없습니다.'
print('프로젝트:', PROJECT_DIR)
%cd $PROJECT_DIR

## 2. 공통 환경 설치

모든 모델을 같은 Transformers 5 환경에서 실행합니다. DeepSeek는 공식 가중치의 Transformers-native 변환본을 사용해 구버전 의존성 충돌을 피합니다. 설치 직후에는 Pillow 모듈 혼용을 막기 위해 런타임이 한 번 자동 재시작됩니다. 재연결되면 `런타임 → 모두 실행`을 다시 누르세요.

In [ ]:
import os
import subprocess
import sys

DEPS_MARKER = Path('/content/.local_modern_ocr_deps_v2')
if not DEPS_MARKER.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'transformers>=5.3.1,<6', 'accelerate>=1.13',
        'sentencepiece', 'safetensors', 'einops', 'timm', 'pandas',
    ])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
        '--no-cache-dir', 'Pillow==12.3.0',
    ])
    DEPS_MARKER.write_text('transformers5-pillow12.3.0', encoding='utf-8')
    print('설치 완료. Pillow 모듈을 깨끗하게 다시 읽도록 런타임을 자동 재시작합니다.')
    print('재연결된 뒤 런타임 → 모두 실행을 다시 누르세요.')
    os.kill(os.getpid(), 9)
else:
    print('설치 완료 표식 확인:', DEPS_MARKER.read_text(encoding='utf-8'))

In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import json
import shutil
import PIL
import torch
import transformers
from PIL import Image
from packaging.version import Version
from transformers import (
    AutoModelForImageTextToText, AutoProcessor,
    LightOnOcrForConditionalGeneration, LightOnOcrProcessor,
    Qwen3VLForConditionalGeneration,
)

assert Version(transformers.__version__).major >= 5, 'Transformers 5가 아닙니다.'
Image.new('RGB', (1, 1), 'white').getpixel((0, 0))
assert torch.cuda.is_available(), '런타임 → 런타임 유형 변경에서 GPU를 선택하세요.'
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
disk = shutil.disk_usage('/content')
print('GPU:', torch.cuda.get_device_name(0), f'{gpu_gb:.1f}GB')
print('Transformers:', transformers.__version__, '| Pillow:', PIL.__version__)
print('Colab 여유 디스크:', f'{disk.free / 1024**3:.1f}GB')
assert gpu_gb >= 14, 'VRAM 14GB 이상을 권장합니다.'

## 3. 데이터와 실행 옵션

첫 실행은 `FIELD_LIMIT=8`로 다섯 모델이 정상 동작하는지만 확인합니다. 성공 후 `FIELD_LIMIT=None`으로 바꾸고 모델 셀부터 다시 실행하면 기존 8개를 재사용해 192개까지 이어집니다.

In [ ]:
def jsonl_count(path):
    with Path(path).open(encoding='utf-8') as handle:
        return sum(1 for line in handle if line.strip())

FIELD_MANIFEST = Path('data/field_manifest.jsonl')
assert jsonl_count(FIELD_MANIFEST) == 192

RUN_PADDLE = True
RUN_GLM = True
RUN_LIGHTON = True
RUN_DEEPSEEK = True
RUN_QWEN = True
FIELD_LIMIT = 8       # 전체 비교 때 None으로 변경
RESET_RESULTS = False # 오류 결과를 지우고 재시도할 때만 True

print({
    'paddle': RUN_PADDLE, 'glm': RUN_GLM, 'lighton': RUN_LIGHTON,
    'deepseek': RUN_DEEPSEEK, 'qwen': RUN_QWEN, 'limit': FIELD_LIMIT,
})

## 4. 공통 벤치마크 러너

In [ ]:
import gc
import sys
from collections import Counter

sys.path.insert(0, str(PROJECT_DIR))
from local_modern_ocr_adapters import (
    DeepSeekOCR2Adapter, GlmOCRAdapter, LightOnOCR2Adapter, Qwen3VLOCRAdapter,
)
try:
    from local_modern_ocr_adapters import PaddleOCRVL16Adapter
except ImportError:
    # Drive에 이전 어댑터 파일이 남아 있어도 새 노트북 단독으로 실행되게 한다.
    from local_modern_ocr_adapters import (
        _ClosableAdapter, _decode_new_tokens, _move_inputs, _runtime,
    )

    class PaddleOCRVL16Adapter(_ClosableAdapter):
        name = 'PaddleOCR-VL-1.6-local'
        model_id = 'PaddlePaddle/PaddleOCR-VL-1.6'

        def __init__(self, max_new_tokens=512):
            from transformers import AutoModelForImageTextToText, AutoProcessor
            self.torch, self.device, self.dtype = _runtime()
            self.max_new_tokens = max_new_tokens
            self.processor = AutoProcessor.from_pretrained(self.model_id)
            self.model = AutoModelForImageTextToText.from_pretrained(
                self.model_id, torch_dtype=self.dtype, low_cpu_mem_usage=True,
            ).to(self.device).eval()

        def predict(self, image_path, unit=None):
            del unit
            image = Image.open(image_path).convert('RGB')
            messages = [{
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': image},
                    {'type': 'text', 'text': 'OCR:'},
                ],
            }]
            inputs = self.processor.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors='pt',
            )
            moved = _move_inputs(inputs, self.device, self.dtype)
            with self.torch.inference_mode():
                output = self.model.generate(
                    **moved, do_sample=False, max_new_tokens=self.max_new_tokens,
                )
            return _decode_new_tokens(self.processor, moved, output)

    print('Drive의 어댑터가 이전 버전이라 Paddle 클래스를 노트북에서 보완했습니다.')
from soapbench.dataset import read_jsonl
from soapbench.runner import run_inference

RUN_DIR = Path('runs/local-modern-ocr')
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS = {
    'paddle': RUN_DIR / 'paddleocr-vl16-field.jsonl',
    'glm': RUN_DIR / 'glm-ocr-field.jsonl',
    'lighton': RUN_DIR / 'lightonocr2-field.jsonl',
    'deepseek': RUN_DIR / 'deepseek-ocr2-field.jsonl',
    'qwen': RUN_DIR / 'qwen3vl-4b-field.jsonl',
}
MODEL_FAILURES = {}

if RESET_RESULTS:
    for path in OUTPUTS.values():
        path.unlink(missing_ok=True)

def read_rows(path):
    return list(read_jsonl(path)) if Path(path).exists() else []

def run_verified(adapter, output):
    output = Path(output)
    total = jsonl_count(FIELD_MANIFEST)
    expected = min(total, FIELD_LIMIT) if FIELD_LIMIT is not None else total
    previous = read_rows(output)
    if previous:
        if any(row.get('model') != adapter.name for row in previous):
            raise RuntimeError(f'{output}에 다른 모델 결과가 있습니다. RESET_RESULTS=True로 재실행하세요.')
        errors = [row for row in previous if row.get('error')]
        if errors:
            raise RuntimeError(f'{output}에 이전 오류가 있습니다: {errors[0]["error"]}')
    else:
        smoke = run_inference(
            manifest=FIELD_MANIFEST, adapter=adapter, output=output, limit=1, resume=False
        )
        first = read_rows(output)[0]
        if smoke['failed'] or first.get('error'):
            raise RuntimeError(f'1개 시험 실패: {first.get("error")}')
        print(adapter.name, '1개 시험 성공')

    result = run_inference(
        manifest=FIELD_MANIFEST, adapter=adapter, output=output,
        limit=FIELD_LIMIT, resume=True,
    )
    rows = read_rows(output)
    errors = [row for row in rows if row.get('error')]
    assert len(rows) == expected, f'{output}: {len(rows)}/{expected}개 저장'
    if errors:
        counts = Counter(row['error'] for row in errors)
        raise RuntimeError(f'추론 오류 {len(errors)}개: {counts.most_common(3)}')
    print(adapter.name, result, '총 표본=', len(rows))
    return output

def run_one(key, factory):
    adapter = None
    try:
        print('\n===', key, '모델 로딩 ===')
        adapter = factory()
        return run_verified(adapter, OUTPUTS[key])
    except Exception as exc:
        MODEL_FAILURES[key] = f'{type(exc).__name__}: {exc}'
        print('실패:', key, MODEL_FAILURES[key])
        return None
    finally:
        if adapter is not None and hasattr(adapter, 'close'):
            adapter.close()
        del adapter
        gc.collect()
        torch.cuda.empty_cache()

## 5. PaddleOCR-VL 1.6 — 0.9B / Apache-2.0

In [ ]:
if RUN_PADDLE:
    run_one('paddle', PaddleOCRVL16Adapter)

## 6. GLM-OCR — 0.9B / MIT

한국어가 명시된 최신 경량 후보라 이번 비교에서 특히 중요합니다.

In [ ]:
if RUN_GLM:
    run_one('glm', GlmOCRAdapter)

## 7. LightOnOCR-2 — 1B / Apache-2.0

공개 벤치마크와 속도는 강하지만 한국어 학습 범위가 명확하지 않아 실제 결과로 탈락 여부를 판단합니다.

In [ ]:
if RUN_LIGHTON:
    run_one('lighton', LightOnOCR2Adapter)

## 8. DeepSeek-OCR-2 — 3B / Apache-2.0

공식 체크포인트는 Transformers 4.46.3과 FlashAttention 2.7.3을 고정하므로, 동일 런타임 비교를 위해 Hugging Face의 Transformers-native 변환본을 사용합니다.

In [ ]:
if RUN_DEEPSEEK:
    run_one('deepseek', DeepSeekOCR2Adapter)

## 9. Qwen3-VL-4B-Instruct — 4B / Apache-2.0

OCR 전용 모델은 아니지만 32개 언어 OCR을 지원하는 강한 일반 VLM 기준선입니다. 교정·요약 금지 프롬프트를 고정합니다.

In [ ]:
if RUN_QWEN:
    run_one('qwen', Qwen3VLOCRAdapter)

## 10. 성공 결과 검증과 공통 리포트

In [ ]:
import subprocess
import pandas as pd
from IPython.display import HTML, display

expected = min(192, FIELD_LIMIT) if FIELD_LIMIT is not None else 192
prediction_files = []
for key, path in OUTPUTS.items():
    rows = read_rows(path)
    errors = [row for row in rows if row.get('error')]
    if len(rows) == expected and not errors:
        prediction_files.append(path)
        print('포함:', key, len(rows))
    else:
        print('제외:', key, '표본=', len(rows), '오류=', len(errors), MODEL_FAILURES.get(key, ''))
if not prediction_files:
    details = json.dumps(MODEL_FAILURES, ensure_ascii=False, indent=2)
    raise RuntimeError(f'모든 모델이 실패했습니다. 위 모델별 오류와 아래 요약을 확인하세요.\n{details}')

REPORT_DIR = Path('reports/local-modern-ocr') / ('full-192' if FIELD_LIMIT is None else f'smoke-{FIELD_LIMIT}')
cmd = [sys.executable, '-m', 'soapbench', 'evaluate', '--predictions']
cmd += [str(path) for path in prediction_files]
cmd += ['--output-dir', str(REPORT_DIR)]
subprocess.run(cmd, check=True)

summary = pd.read_csv(REPORT_DIR / 'summary.csv')
summary['screen_pass'] = (
    (summary['cer_no_whitespace'] <= 0.05)
    & (summary['critical_accuracy'] >= 0.99)
    & (summary['omission_rate'] == 0)
)
display(summary.sort_values(['screen_pass', 'cer_no_whitespace', 'critical_accuracy'], ascending=[False, True, False]))
display(HTML(filename=str(REPORT_DIR / 'report.html')))

`screen_pass`는 합성 데이터 후보 압축용일 뿐 배포 승인값이 아닙니다. 실제 익명화 필기 30~50개에서 숫자·날짜·위기 문구·부정 표현을 별도로 확인해야 합니다.

## 11. 결과 ZIP 다운로드

In [ ]:
import tempfile
from google.colab import files

bundle = Path(tempfile.mkdtemp(prefix='local_ocr_benchmark_'))
shutil.copytree(REPORT_DIR, bundle / 'report')
(bundle / 'runs').mkdir()
for path in prediction_files:
    shutil.copy2(path, bundle / 'runs' / path.name)
shutil.copy2('LOCAL_OCR_MODEL_NOTES.md', bundle / 'LOCAL_OCR_MODEL_NOTES.md')
archive = shutil.make_archive('/content/local_modern_ocr_results', 'zip', bundle)
print('다운로드:', archive)
files.download(archive)